# LongCoT task benchmark for Fleet

Load a small, reproducible task dataset from Hugging Face, run Fleet through its standard API, and inspect official correctness alongside Fleet's own evaluation metrics. Default: ten easy tasks, two from each of five domains. This is a diagnostic sample, not a leaderboard reproduction.

Install notebook tooling once with `uv sync --locked --group benchmarks`, and select this repository's `.venv` kernel. Task preparation downloads public data; live execution and Fleet judges have separate disabled gates.

LongCoT's HF export omits structured verifier payloads for some domains. Join selected HF tasks to the pinned upstream verification package by question ID, requiring identical prompts. Gold and verifier payloads stay in host-side receipts. The published prompt is used verbatim; no benchmark guidance is injected. Historical runs under `outputs/longcot/` remain separate.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "pyproject.toml").is_file(), "Open this notebook from the Fleet repository"
sys.path.insert(0, str(ROOT / "scripts"))

In [ ]:
import hashlib
import json

import pandas as pd
from benchmark_tasks import evaluate_saved_tasks, freeze_tasks, load_tasks, run_tasks, score_tasks
from datasets import load_dataset

from fleet_rlm.config import get_settings

MLFLOW_MODE = "local"  # Or "databricks", matching the running Fleet service.
settings = get_settings(MLFLOW_MODE)
BASE_URL = f"http://{settings.host}:{settings.port}"
ALLOW_LIVE = False
ALLOW_JUDGES = False

In [ ]:
DATASET = "LongHorizonReasoning/longcot"
REVISION = "7f9948570f5239f0f3697c6c9793b4866860c681"
SPLIT = "easy"
PER_DOMAIN = 2
SEED = 20261021
BUNDLE = ROOT / "outputs/benchmarks/longcot-easy-v1"
RUN_DIR = BUNDLE / "runs/run-001"  # Change for an intentional new measurement.

In [ ]:
from dataclasses import asdict
from importlib.metadata import distribution

import longcot

VERIFIER_COMMIT = "fb9649423f15f5b0091f8e988b100596cac592ca"
DOMAINS = ("logic", "cs", "chemistry", "chess", "math")


def require_verifier(expected=VERIFIER_COMMIT):
    """Require the installed LongCoT Git commit to match ``expected``.

    Raise ValueError on a mismatch; package lookup and metadata decoding errors propagate.
    """
    source = json.loads(distribution("longcot").read_text("direct_url.json") or "{}")
    if source.get("vcs_info", {}).get("commit_id") != expected:
        raise ValueError("Installed longcot differs from the required commit")


def select_longcot(rows, per_domain=2, seed=20261021):
    """Choose a deterministic, seeded sample from each domain.

    Request ``per_domain`` rows per domain, returning them in DOMAINS order.
    Raise ValueError if any domain has fewer rows than requested.
    """
    chosen = []
    for domain in DOMAINS:
        pool = [row for row in rows if row["domain"] == domain]
        pool.sort(key=lambda row: hashlib.sha256(f"{seed}:{row['question_id']}".encode()).hexdigest())
        if len(pool) < per_domain:
            raise ValueError(f"Insufficient {domain} tasks; refusing to change the sample")
        chosen.extend(pool[:per_domain])
    return chosen


def longcot_task(row, canonical):
    """Build a task with the canonical verifier payload and any host-only gold answer.

    ``canonical`` maps question IDs to LongCoT questions. Raise ValueError if prompt, domain,
    or difficulty differs from the HF row; a missing canonical question raises KeyError.
    """
    question = canonical[row["question_id"]]
    if (question.prompt, question.domain, question.difficulty) != (row["prompt"], row["domain"], row["difficulty"]):
        raise ValueError("HF task differs from the pinned verifier's canonical question")
    payload = asdict(question)
    return {
        "case_id": row["question_id"],
        "task": row["prompt"],
        "question": payload,
        "source_group": f"{row['domain']}:{row['template']}",
        "expected_answer": json.dumps(question.answer) if question.answer is not None else None,
    }


def official_score(case, answer):
    """Return 1.0 for a nonblank answer accepted by the pinned verifier, otherwise 0.0.

    Disable math and chemistry model fallbacks. Verifier pin and verification errors propagate.
    """
    require_verifier(bundle["metadata"]["verifier_commit"])
    options = longcot.VerifyOptions(
        math=longcot.MathVerifyOptions(enable_fallback=False),
        chemistry=longcot.ChemistryVerifyOptions(enable_fallback=False),
    )
    return float(bool(answer.strip()) and longcot.verify(longcot.Question(**case["question"]), answer, options=options))

In [ ]:
if BUNDLE.exists():
    bundle = load_tasks(BUNDLE)
else:
    require_verifier()
    source = load_dataset(DATASET, "all", split=SPLIT, revision=REVISION)
    rows = select_longcot(list(source), PER_DOMAIN, SEED)
    canonical = {question.question_id: question for question in longcot.load_questions(difficulty=SPLIT)}
    bundle = freeze_tasks(
        BUNDLE,
        [longcot_task(row, canonical) for row in rows],
        {
            "dataset": DATASET,
            "dataset_revision": REVISION,
            "split": SPLIT,
            "dataset_url": f"https://huggingface.co/datasets/{DATASET}/tree/{REVISION}",
            "seed": SEED,
            "per_domain": PER_DOMAIN,
            "verifier_commit": VERIFIER_COMMIT,
            "protocol": "Published prompts, standard Fleet settings, no verifier model fallback",
        },
    )
display(pd.DataFrame([{"case_id": case["case_id"], "source_group": case["source_group"]} for case in bundle["cases"]]))
print("Frozen task digest:", bundle["digest"])

Run Fleet on the frozen tasks. Start the normal service yourself (`uv run fleet cli`, or `uv run fleet cli databricks`). Set `ALLOW_LIVE=True` only when ready for model and Daytona usage. The helper captures the effective configuration without changing models, mode, retries or token limits. Failed/uncertain receipts block replay; inspect their sessions and saved outcomes before choosing a new run.

In [ ]:
if ALLOW_LIVE:
    run_tasks(BUNDLE, RUN_DIR, base_url=BASE_URL, expected_destination=settings.mlflow_descriptor(), allow_live=True)
else:
    print("Live execution disabled; frozen task preparation and saved-output scoring are separate.")

Score existing receipts independently of execution. Official verification below is local, with model fallback disabled. Completed-case accuracy excludes operational failures; all-selected-case accuracy includes failures and unattempted cases in its denominator. Partial results are marked incomplete.

In [ ]:
if (RUN_DIR / "run.json").exists():
    report = score_tasks(BUNDLE, RUN_DIR, official_score)
    display(pd.Series({key: value for key, value in report.items() if key != "cases"}))
    display(pd.json_normalize(report["cases"]))
else:
    print("No saved run yet.")

Evaluate the saved traces with Fleet's existing MLflow scorer suite and its DSPy judge, plus the official benchmark score as a separate assessment. This uses the configured evaluation experiment, records the source dataset and all selected task IDs, and labels traces `benchmark` so optimization curation excludes them. Set `ALLOW_JUDGES=True` deliberately: judges incur model usage. Trace IDs and the MLflow evaluation run remain available even if later scoring fails; re-score saved traces without executing tasks again.

In [ ]:
if ALLOW_JUDGES:
    evaluation = evaluate_saved_tasks(BUNDLE, RUN_DIR, settings, allow_judges=True)
    print("MLflow evaluation run:", evaluation.run_id)
    display(evaluation.result_df)
else:
    print("Fleet judge evaluation disabled.")